In [1]:
# ═══════════════════════════════════════════════════════════════
# CELL 1: IMPORTS & SETUP (Groq API via HTTP)
# ═══════════════════════════════════════════════════════════════
import os
import json
import time
import requests
from typing import TypedDict, List, Optional, Dict, Any
from datetime import datetime

# ── Groq API Config ──
GROQ_API_KEY = "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"
GROQ_API_URL = "https://api.groq.com/openai/v1/chat/completions"
GROQ_MODEL   = "meta-llama/llama-4-scout-17b-16e-instruct"

# ── Retry Config ──
MAX_RETRIES = 6
BASE_DELAY  = 20

# ── Token Tracker ──
_tracker = {"prompt": 0, "completion": 0, "calls": 0}

# Giá llama-4-scout-17b (USD / 1M tokens) — kiểm tra lại trên console.groq.com
PRICE_INPUT  = 0.11
PRICE_OUTPUT = 0.34

def reset_tracker():
    _tracker["prompt"] = _tracker["completion"] = _tracker["calls"] = 0

def print_token_summary():
    p, c, n = _tracker["prompt"], _tracker["completion"], _tracker["calls"]
    cost = (p * PRICE_INPUT + c * PRICE_OUTPUT) / 1_000_000
    print(f"""
╔══════════════════════════════════════╗
║       TOKEN & COST SUMMARY           ║
╠══════════════════════════════════════╣
║  API calls  : {n:>6}                 ║
║  Prompt     : {p:>6} tokens          ║
║  Completion : {c:>6} tokens          ║
║  Total      : {p+c:>6} tokens        ║
║  Est. cost  : ${cost:.6f} USD        ║
╚══════════════════════════════════════╝""")

def groq_chat(system_prompt: str, user_prompt: str,
              temperature: float = 0.6, max_tokens: int = 4096) -> str:
    """Gọi Groq API qua HTTP, tự retry khi 429/503"""
    headers = {
        "Authorization": f"Bearer {GROQ_API_KEY}",
        "Content-Type": "application/json",
    }
    payload = {
        "model": GROQ_MODEL,
        "messages": [
            {"role": "system", "content": system_prompt},
            {"role": "user",   "content": user_prompt},
        ],
        "temperature": temperature,
        "max_completion_tokens": max_tokens,
        "top_p": 0.95,
        "stream": False,
    }

    for attempt in range(MAX_RETRIES):
        response = requests.post(GROQ_API_URL, headers=headers, json=payload, timeout=120)

        if response.status_code == 200:
            data    = response.json()
            usage   = data.get("usage", {})
            p_tok   = usage.get("prompt_tokens", 0)
            c_tok   = usage.get("completion_tokens", 0)

            # cộng vào tracker
            _tracker["prompt"]     += p_tok
            _tracker["completion"] += c_tok
            _tracker["calls"]      += 1

            print(f"  ✓ +{p_tok}p +{c_tok}c tokens "
                  f"(total {_tracker['prompt']+_tracker['completion']})")
            return data["choices"][0]["message"]["content"]

        elif response.status_code in (429, 503):
            retry_after = response.headers.get("Retry-After")
            wait = int(retry_after) if retry_after else BASE_DELAY * (2 ** attempt)
            print(f"  ⏳ {response.status_code} — chờ {wait}s "
                  f"(lần {attempt+1}/{MAX_RETRIES})...")
            time.sleep(wait)

        else:
            response.raise_for_status()

    raise RuntimeError(f"❌ Thất bại sau {MAX_RETRIES} lần thử.")

print("✅ Cell 1: Setup hoàn tất")

✅ Cell 1: Setup hoàn tất


In [2]:
# ═══════════════════════════════════════════════════════════════
# CELL 2: STATE DEFINITION (PAS + AIDA + StoryBrand Hybrid)
# ═══════════════════════════════════════════════════════════════

class ContentFactoryState(TypedDict):
    client_brief: Dict[str, Any]
    content_brief: Optional[Dict[str, Any]]
    research_data: Optional[Dict[str, Any]]
    long_form_draft: Optional[str]
    social_copies: Optional[Dict[str, Any]]
    visual_prompts: Optional[List[str]]
    seo_optimized_content: Optional[Dict[str, Any]]
    qa_score: Optional[float]
    qa_feedback: Optional[str]
    approved_content: Optional[Dict[str, Any]]
    distribution_plan: Optional[Dict[str, Any]]
    performance_metrics: Optional[Dict[str, Any]]
    optimization_notes: Optional[str]
    current_step: str
    retry_count: int

# Test brief mẫu
TEST_BRIEF = {
    "brand_name": "TechStart Vietnam",
    "product": "Phần mềm quản lý dự án AI cho startup",
    "target_audience": "Founder startup Việt Nam, 25-40 tuổi, tech-savvy",
    "goal": "Tăng sign-up trial 200% trong Q3",
    "tone": "Chuyên nghiệp nhưng gần gũi, data-driven, có chút humor",
    "channels": ["blog", "linkedin", "email", "twitter"],
    "content_type": "long_form_blog",
    "word_count": 2500,
    "keywords": ["AI project management", "startup tools Vietnam", "quản lý dự án AI"],
    "competitors": ["Notion", "Asana", "Monday.com"],
    "brand_voice_examples": [
        "Chúng tôi không bán phần mềm. Chúng tôi bán thời gian.",
        "Startup không cần thêm công cụ. Startup cần ít công cụ hơn, hiệu quả hơn."
    ]
}

print("✅ Cell 2: State & Test Brief định nghĩa xong")

✅ Cell 2: State & Test Brief định nghĩa xong


In [3]:
# ═══════════════════════════════════════════════════════════════
# CELL 2.5: MOCK DATA — Competitor Research + Brand Voice RAG
# Thay thế tạm cho real scraper + vector DB
# ═══════════════════════════════════════════════════════════════

# ── Mock 1: Competitor SERP Research ──
MOCK_COMPETITOR_DATA = {
    "keyword": "phần mềm quản lý dự án",
    "serp_top10": [
        {
            "rank": 1, "url": "https://asana.com/vi",
            "title": "Asana: Quản lý công việc nhóm hiệu quả",
            "word_count": 2400,
            "headings": ["Tại sao chọn Asana?", "Tính năng nổi bật", "Bắt đầu miễn phí"],
            "pain_points": ["deadline trễ", "team không sync", "báo cáo thủ công"],
            "cta": "Dùng thử miễn phí 30 ngày"
        },
        {
            "rank": 2, "url": "https://monday.com/vi",
            "title": "monday.com — Nền tảng quản lý việc làm",
            "word_count": 1900,
            "headings": ["Quản lý mọi dự án", "Tích hợp 200+ app", "Báo cáo realtime"],
            "pain_points": ["khó theo dõi tiến độ", "quá nhiều tool rời rạc", "tốn thời gian họp"],
            "cta": "Bắt đầu miễn phí"
        },
        {
            "rank": 3, "url": "https://notion.so/vi",
            "title": "Notion — All-in-one workspace",
            "word_count": 1600,
            "headings": ["Docs + Tasks + Wiki", "Cho team mọi quy mô", "Template sẵn có"],
            "pain_points": ["thông tin phân tán", "không có workflow chuẩn", "onboarding lâu"],
            "cta": "Dùng Notion miễn phí"
        }
    ],
    "avg_word_count": 1967,
    "common_sections": ["Hook/Problem", "Tính năng", "Social Proof", "Pricing", "CTA"],
    "content_gaps": [
        "Không ai nói về ROI cụ thể cho startup Việt",
        "Thiếu case study tiếng Việt",
        "Không có so sánh chi phí bằng VND"
    ]
}

# ── Mock 2: Brand Voice RAG ──
MOCK_BRAND_VOICE = {
    "brand_name": "ProjectFlow",
    "retrieved_chunks": [
        {
            "source": "website_homepage.txt", "score": 0.95,
            "content": "ProjectFlow giúp founder startup Việt kiểm soát dự án như một CEO thực thụ — không cần MBA, không cần team lớn."
        },
        {
            "source": "sample_blog_post.txt", "score": 0.91,
            "content": "Chúng tôi không bán phần mềm. Chúng tôi bán sự yên tâm khi bạn biết team đang đi đúng hướng."
        },
        {
            "source": "social_media_posts.txt", "score": 0.88,
            "content": "Tone: thẳng thắn, không hoa mỹ. Dùng số liệu cụ thể. Tránh buzzword. Nói chuyện như người bạn có chuyên môn."
        },
        {
            "source": "email_campaign.txt", "score": 0.85,
            "content": "CTA luôn rõ ràng, một hành động duy nhất. Không bao giờ dùng 'Tìm hiểu thêm' — dùng 'Xem demo 5 phút' hoặc 'Thử ngay hôm nay'."
        }
    ],
    "voice_summary": "Thẳng thắn, dùng số liệu, không hoa mỹ, gần gũi như người bạn có chuyên môn"
}

def mock_competitor_research(keyword: str = None) -> dict:
    """Giả lập SERP scraper — trả về competitor data"""
    print(f"  [MOCK] Competitor research: {len(MOCK_COMPETITOR_DATA['serp_top10'])} competitors")
    return MOCK_COMPETITOR_DATA

def mock_brand_voice_rag(query: str = None) -> dict:
    """Giả lập vector DB retrieval — trả về brand voice chunks"""
    print(f"  [MOCK] RAG retrieved: {len(MOCK_BRAND_VOICE['retrieved_chunks'])} chunks")
    return MOCK_BRAND_VOICE

print("✅ Cell 2.5: Mock Competitor + Brand Voice RAG sẵn sàng")

✅ Cell 2.5: Mock Competitor + Brand Voice RAG sẵn sàng


In [4]:
# ═══════════════════════════════════════════════════════════════
# CELL 3: NODE 1 — STRATEGIST AGENT
# Framework: StoryBrand (Donald Miller) + PAS Problem Identification
# ═══════════════════════════════════════════════════════════════

STRATEGIST_SYSTEM_PROMPT = """Bạn là Chief Content Strategist đào tạo bởi StoryBrand (Donald Miller).
Áp dụng framework: StoryBrand 7-part framework + PAS Problem identification.

STORYBRAND 7 PHẦN:
1. A CHARACTER (Hero): Khách hàng là hero, không phải brand
2. HAS A PROBLEM: External, Internal, Philosophical problems
3. AND MEETS A GUIDE: Brand là guide, không phải hero
4. WHO HAS A PLAN: 3-step plan rõ ràng
5. AND CALLS THEM TO ACTION: Direct CTA + Transitional CTA
6. THAT HELPS THEM AVOID FAILURE: Stakes nếu không hành động
7. AND ENDS IN A SUCCESS: Transformation vision

PAS PROBLEM IDENTIFICATION:
- Problem: Pain point rõ ràng, specific
- Agitate: Amplify pain, make it emotional, add urgency
- Solution: Position product as giải pháp duy nhất

OUTPUT JSON:
{
  "storybrand_framework": {
    "hero": "mô tả khách hàng (không phải brand)",
    "external_problem": "vấn đề bề mặt",
    "internal_problem": "vấn đề cảm xúc/sâu hơn",
    "philosophical_problem": "vấn đề lớn hơn cuộc sống",
    "guide": "brand là guide như thế nào",
    "empathy": "brand hiểu hero",
    "authority": "brand có credentials gì",
    "plan": ["Bước 1", "Bước 2", "Bước 3"],
    "direct_cta": "hành động chính",
    "transitional_cta": "hành động nhỏ hơn",
    "failure_stakes": "điều gì xảy ra nếu không hành động",
    "success_vision": "transformation sau khi dùng sản phẩm"
  },
  "pas_framework": {
    "problem_statement": "vấn đề rõ ràng",
    "agitation_points": ["điểm làm đau 1", "điểm làm đau 2", "điểm làm đau 3"],
    "solution_intro": "giới thiệu sản phẩm như giải pháp"
  },
  "title": "tiêu đề theo StoryBrand + PAS",
  "content_angle": "điểm khác biệt",
  "keyword_cluster": {"primary": "...", "secondary": [...], "lsi": [...]},
  "outline": [
    {"section": "Hook (AIDA Attention)", "word_count": 150, "framework": "AIDA", "key_points": [...]},
    {"section": "Problem (PAS)", "word_count": 300, "framework": "PAS", "key_points": [...]},
    {"section": "Agitate (PAS)", "word_count": 400, "framework": "PAS", "key_points": [...]},
    {"section": "Meet the Guide (StoryBrand)", "word_count": 300, "framework": "StoryBrand", "key_points": [...]},
    {"section": "The Plan (StoryBrand)", "word_count": 400, "framework": "StoryBrand", "key_points": [...]},
    {"section": "Proof (AIDA Desire)", "word_count": 500, "framework": "AIDA", "key_points": [...]},
    {"section": "CTA (AIDA Action + StoryBrand)", "word_count": 200, "framework": "AIDA+StoryBrand", "key_points": [...]},
    {"section": "Success Vision (StoryBrand)", "word_count": 250, "framework": "StoryBrand", "key_points": [...]}
  ],
  "kpis": {"traffic_target": N, "conversion_target": N, "engagement_target": N}
}
"""

def strategist_node(state: ContentFactoryState):
    """Node 1: StoryBrand + PAS Strategy"""
    brief = state["client_brief"]
    
    user_prompt = f"""CLIENT BRIEF:
Brand: {brief['brand_name']}
Product: {brief['product']}
Target Audience: {brief['target_audience']}
Goal: {brief['goal']}
Tone: {brief['tone']}
Keywords: {', '.join(brief['keywords'])}
Competitors: {', '.join(brief['competitors'])}
Brand Voice: {brief['brand_voice_examples']}

Áp dụng StoryBrand + PAS. CHỈ trả về JSON."""
    
    # ← CHỈ SỬA DÒNG NÀY: 4096 → 600
    response = groq_chat(STRATEGIST_SYSTEM_PROMPT, user_prompt, temperature=0.7, max_tokens=1200)
    
    try:
        content_brief = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        # ← VÀ DÒNG NÀY: tránh lưu response dài khi lỗi
        content_brief = {"raw_response": response[:300], "parse_error": True}
    
    return {**state, "content_brief": content_brief, "current_step": "research"}

print("✅ Cell 3: Strategist Node (StoryBrand + PAS) định nghĩa xong")

✅ Cell 3: Strategist Node (StoryBrand + PAS) định nghĩa xong


In [5]:
# ═══════════════════════════════════════════════════════════════
# CELL 4: NODE 2 — RESEARCHER AGENT
# Focus: Customer pain points, competitor gaps, proof data
# ═══════════════════════════════════════════════════════════════
RESEARCHER_SYSTEM_PROMPT = """Bạn là Senior Research Analyst chuyên pain point mining.
Nhiệm vụ: Thu thập dữ liệu để SUPPORT StoryBrand + PAS framework.

OUTPUT JSON:
{
  "pain_points": [
    {"pain": "...", "frequency": "high|medium|low", "emotional_intensity": 8, "source": "..."}
  ],
  "competitor_gaps": [
    {"competitor": "...", "gap": "...", "opportunity": "..."}
  ],
  "proof_data": {
    "statistics": [{"claim": "...", "source": "...", "year": 2024}],
    "case_studies": [{"company": "...", "before": "...", "after": "...", "metric": "..."}],
    "testimonials": [{"quote": "...", "persona": "..."}]
  },
  "emotional_triggers": ["trigger 1", "trigger 2"],
  "common_objections": [{"objection": "...", "reframe": "..."}],
  "voice_of_customer": ["cách khách hàng nói 1", "cách nói 2"]
}"""

def researcher_node(state: ContentFactoryState):
    """Node 2: Pain Point Mining & Proof Gathering"""
    brief  = state.get("content_brief") or {}
    client = state["client_brief"]

    # ── Lấy mock data (swap sang real system khi sẵn sàng) ──
    competitor_data = mock_competitor_research()
    brand_voice     = mock_brand_voice_rag()

    storybrand = brief.get("storybrand_framework", {})

    user_prompt = f"""STORYBRAND CONTEXT:
Hero: {storybrand.get('hero', 'N/A')}
Problem: {storybrand.get('external_problem', 'N/A')}

COMPETITOR INSIGHTS (từ SERP):
- Content gaps: {competitor_data['content_gaps']}
- Common pain points: {[p for c in competitor_data['serp_top10'] for p in c['pain_points']][:6]}

BRAND VOICE (RAG):
{chr(10).join([f"- {c['content']}" for c in brand_voice['retrieved_chunks']])}

COMPETITORS: {', '.join(client.get('competitors', []))}

Dựa vào data trên, tổng hợp pain points và proof data. CHỈ trả về JSON."""

    response = groq_chat(RESEARCHER_SYSTEM_PROMPT, user_prompt, temperature=0.6, max_tokens=800)

    try:
        research_data = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        research_data = {"raw_response": response[:300], "parse_error": True}

    return {**state, "research_data": research_data, "current_step": "creation"}

print("✅ Cell 4: Researcher Node (Pain Point Mining) định nghĩa xong")

✅ Cell 4: Researcher Node (Pain Point Mining) định nghĩa xong


In [6]:
# ═══════════════════════════════════════════════════════════════
# CELL 5: NODE 3 — CREATION AGENT
# Framework: PAS → StoryBrand Guide → AIDA Desire → StoryBrand Success
# ═══════════════════════════════════════════════════════════════

WRITER_SYSTEM_PROMPT = """Bạn là Conversion Copywriter. Viết content theo cấu trúc:

SECTION 1: HOOK (AIDA Attention)
SECTION 2: PROBLEM (PAS - external + internal + philosophical)
SECTION 3: AGITATE (PAS - amplify pain, urgency)
SECTION 4: MEET THE GUIDE (StoryBrand - empathy + authority)
SECTION 5: THE PLAN (StoryBrand - 3 bước)
SECTION 6: PROOF (AIDA Desire - case study + testimonials)
SECTION 7: CTA (AIDA Action + StoryBrand)
SECTION 8: SUCCESS VISION (StoryBrand transformation)

QUY TẮC: Dùng "Bạn" nhiều hơn "Chúng tôi". Mỗi paragraph max 3 dòng."""

COPYWRITER_SYSTEM_PROMPT = """Bạn là Social Media Copywriter.
Tạo social copies từ long-form content.
OUTPUT JSON: {"linkedin": "...", "twitter": "...", "instagram": "...", "email": {"subject": "...", "body": "..."}}"""

DESIGNER_SYSTEM_PROMPT = """Bạn là Visual Content Strategist.
Tạo image prompts theo StoryBrand visual rules (hero là khách hàng, không phải sản phẩm).
OUTPUT JSON list: [{"section": "...", "prompt": "...", "style": "..."}]"""

def creation_node(state: ContentFactoryState):
    """Node 3: PAS + AIDA + StoryBrand Content Creation"""
    brief    = state.get("content_brief") or {}
    research = state.get("research_data") or {}
    client   = state["client_brief"]

    # ── Extract brand voice + competitor gaps từ research ──
    brand_voice     = research.get("brand_voice", MOCK_BRAND_VOICE)
    competitor_gaps = research.get("competitor_gaps", MOCK_COMPETITOR_DATA.get("content_gaps", []))
    pain_points     = [p["pain"] for p in research.get("pain_points", [])[:4]]
    storybrand      = brief.get("storybrand_framework", {})
    pas             = brief.get("pas_framework", {})

    # 3A: Long-form Writer
    writer_prompt = f"""STORYBRAND:
Hero: {storybrand.get('hero', 'N/A')}
Problem: {storybrand.get('external_problem', 'N/A')} / {storybrand.get('internal_problem', 'N/A')}
Plan: {storybrand.get('plan', [])}
CTA: {storybrand.get('direct_cta', 'N/A')}
Success: {storybrand.get('success_vision', 'N/A')}

PAS:
Problem: {pas.get('problem_statement', 'N/A')}
Agitation: {pas.get('agitation_points', [])}

PAIN POINTS (từ research): {pain_points}

BRAND VOICE RULES (bắt buộc áp dụng):
{chr(10).join([f"- {c['content']}" for c in brand_voice.get('retrieved_chunks', [])])}

CONTENT GAPS CẦN ADDRESS:
{chr(10).join([f"- {g}" for g in competitor_gaps])}

Viết 8 sections. Áp dụng đúng brand voice, address content gaps."""

    long_form = groq_chat(WRITER_SYSTEM_PROMPT, writer_prompt, temperature=0.7, max_tokens=1500)

    # 3B: Social Copywriter
    copy_prompt = f"""CONTENT (tóm tắt):
Title: {brief.get('title', 'N/A')}
Hook: {long_form[:400]}
Brand voice: {brand_voice.get('voice_summary', 'N/A')}

Tạo social copies. CHỈ trả về JSON."""

    copy_response = groq_chat(COPYWRITER_SYSTEM_PROMPT, copy_prompt, temperature=0.7, max_tokens=400)
    try:
        social_copies = json.loads(copy_response.replace("```json", "").replace("```", "").strip())
    except:
        social_copies = {"raw": copy_response[:200]}

    # 3C: Visual Prompts
    design_prompt = f"""Content: {brief.get('title', 'N/A')}
Hero: {storybrand.get('hero', 'N/A')}
Tạo 5 image prompts. CHỈ trả về JSON list."""

    design_response = groq_chat(DESIGNER_SYSTEM_PROMPT, design_prompt, temperature=0.7, max_tokens=400)
    try:
        visual_prompts = json.loads(design_response.replace("```json", "").replace("```", "").strip())
    except:
        visual_prompts = [design_response[:200]]

    return {**state, "long_form_draft": long_form, "social_copies": social_copies,
            "visual_prompts": visual_prompts, "current_step": "seo"}

print("✅ Cell 5: Creation Node (PAS+AIDA+StoryBrand) định nghĩa xong")

✅ Cell 5: Creation Node (PAS+AIDA+StoryBrand) định nghĩa xong


In [7]:
# ═══════════════════════════════════════════════════════════════
# CELL 6: NODE 4 — SEO OPTIMIZER
# ═══════════════════════════════════════════════════════════════

SEO_SYSTEM_PROMPT = """Bạn là Technical SEO Expert.
Optimize content đã viết theo StoryBrand + PAS cho search engines.

SEO RULES:
1. Title: Include primary keyword + power word + number (StoryBrand title optimized)
2. Meta: Include PAS agitation point + CTA
3. Headers: H1 = StoryBrand hero statement, H2s = PAS sections
4. Keywords: Primary trong first 100 words, H1, first H2
5. NLP terms: Semantic keywords từ research
6. Internal links: 3-5 links với anchor text tự nhiên
7. Schema: Article + FAQ schema

OUTPUT JSON:
{
  "title_tag": "...",
  "meta_description": "...",
  "header_structure": [...],
  "keyword_placement": {...},
  "optimized_content": "full content với headers",
  "schema_markup": {...}
}"""

def seo_optimizer_node(state: ContentFactoryState):
    """Node 4: SEO Optimization"""
    draft = state["long_form_draft"]
    brief = state["content_brief"]
    
    user_prompt = f"""CONTENT:
{draft[:4000]}...

PRIMARY KEYWORD: {brief.get('keyword_cluster', {}).get('primary', 'N/A')}

Optimize. CHỈ trả về JSON."""
    
    response = groq_chat(SEO_SYSTEM_PROMPT, user_prompt, temperature=0.5, max_tokens=600)
    
    try:
        seo_data = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        seo_data = {"raw_response": response, "parse_error": True}
    
    return {**state, "seo_optimized_content": seo_data, "current_step": "qa"}

print("✅ Cell 6: SEO Optimizer Node định nghĩa xong")

✅ Cell 6: SEO Optimizer Node định nghĩa xong


In [8]:
# ═══════════════════════════════════════════════════════════════
# CELL 7: NODE 5 — QUALITY ASSURANCE
# Check: StoryBrand compliance + PAS strength + AIDA flow
# ═══════════════════════════════════════════════════════════════

EDITOR_SYSTEM_PROMPT = """Bạn là Chief Editor chuyên StoryBrand + PAS + AIDA.
Đánh giá content theo FRAMEWORK COMPLIANCE.

SCORING (0-10 mỗi tiêu chí):

StoryBrand Compliance:
1. Hero is customer (not brand)? 
2. External/Internal/Philosophical problems clear?
3. Guide positioning correct (empathy + authority)?
4. 3-step plan simple and clear?
5. Direct CTA + Transitional CTA present?
6. Failure stakes mentioned?
7. Success vision compelling?

PAS Compliance:
8. Problem clearly stated and relatable?
9. Agitation amplifies pain effectively?
10. Solution introduced naturally?

AIDA Compliance:
11. Hook grabs attention in 3 seconds?
12. Interest maintained through story?
13. Desire built with proof?
14. Action CTA clear and urgent?

OUTPUT JSON:
{
  "storybrand_score": N,
  "pas_score": N,
  "aida_score": N,
  "overall_score": N.N,
  "verdict": "APPROVED|CONDITIONAL|REJECT",
  "feedback": {"strengths": [...], "weaknesses": [...], "action_items": [...]},
  "revision_needed": true|false
}"""

def qa_node(state: ContentFactoryState):
    """Node 5: Framework Compliance QA"""
    seo_content = state["seo_optimized_content"]
    social = state["social_copies"]
    client = state["client_brief"]
    
    user_prompt = f"""SEO CONTENT: {json.dumps(seo_content, ensure_ascii=False)[:2000]}...
SOCIAL COPIES: {json.dumps(social, ensure_ascii=False)[:1000]}...

Đánh giá framework compliance. CHỈ trả về JSON."""
    
    response = groq_chat(EDITOR_SYSTEM_PROMPT, user_prompt, temperature=0.3, max_tokens=600)
    
    try:
        qa_result = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        qa_result = {"overall_score": 0, "verdict": "REJECT", "raw": response}
    
    overall = qa_result.get("overall_score", 0)
    verdict = qa_result.get("verdict", "REJECT")
    
    if overall >= 8.0 and verdict == "APPROVED":
        return {**state, "qa_score": overall, "qa_feedback": qa_result.get("feedback", {}),
                "approved_content": {"seo": seo_content, "social": social, "visual": state["visual_prompts"]},
                "current_step": "distribute"}
    else:
        return {**state, "qa_score": overall, "qa_feedback": qa_result.get("feedback", {}),
                "current_step": "creation", "retry_count": state.get("retry_count", 0) + 1}

print("✅ Cell 7: QA Node (Framework Compliance) định nghĩa xong")

✅ Cell 7: QA Node (Framework Compliance) định nghĩa xong


In [9]:
# ═══════════════════════════════════════════════════════════════
# CELL 8: NODE 6 — DISTRIBUTION
# ═══════════════════════════════════════════════════════════════

DISTRIBUTOR_SYSTEM_PROMPT = """Bạn là Distribution Strategist.
Tạo distribution plan cho StoryBrand content.

TIMELINE:
Day 0: Blog publish (full StoryBrand story)
Day 1: LinkedIn (Hero-focused, problem agitation)
Day 2: Twitter thread (PAS breakdown, 7 tweets)
Day 3: Email (Personal story, CTA)
Day 7: Repurpose: Short video script (Hook + Plan + CTA)
Day 14: Case study follow-up

OUTPUT JSON:
{
  "distribution_timeline": [...],
  "platform_formatting": {...},
  "ab_test_plan": {...},
  "tracking_setup": {...}
}"""

def distributor_node(state: ContentFactoryState):
    """Node 6: Distribution Planning"""
    approved = state["approved_content"]
    client = state["client_brief"]
    
    user_prompt = f"""APPROVED CONTENT: {json.dumps(approved.get('seo', {}), ensure_ascii=False)[:1500]}...
CHANNELS: {', '.join(client.get('channels', []))}

Tạo plan. CHỈ trả về JSON."""
    
    response = groq_chat(DISTRIBUTOR_SYSTEM_PROMPT, user_prompt, temperature=0.6, max_tokens=600)
    
    try:
        dist_plan = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        dist_plan = {"raw": response}
    
    return {**state, "distribution_plan": dist_plan, "current_step": "analytics"}

print("✅ Cell 8: Distribution Node định nghĩa xong")

✅ Cell 8: Distribution Node định nghĩa xong


In [10]:
# ═══════════════════════════════════════════════════════════════
# CELL 9: NODE 7 — ANALYTICS & FEEDBACK
# ═══════════════════════════════════════════════════════════════

ANALYST_SYSTEM_PROMPT = """Bạn là Marketing Analytics Expert.
Tạo analytics framework cho StoryBrand + PAS content.

KPIs by Framework Element:
- Hook (AIDA Attention): Scroll depth, time on page
- Problem/Agitate (PAS): Comment sentiment, shares
- Guide/Plan (StoryBrand): CTA clicks, trial signups
- Proof (AIDA Desire): Conversion rate
- Success Vision: Email replies, brand mentions

OUTPUT JSON:
{
  "kpis": {...},
  "framework_performance": {
    "storybrand_elements": {"hero_clarity": "...", "guide_appeal": "..."},
    "pas_effectiveness": {"problem_resonance": "...", "agitation_strength": "..."},
    "aida_flow": {"attention_grab": "...", "desire_build": "..."}
  },
  "optimization_opportunities": [...],
  "feedback_for_next_cycle": {
    "storybrand_improvements": [...],
    "pas_improvements": [...],
    "aida_improvements": [...]
  }
}"""

def analytics_node(state: ContentFactoryState):
    """Node 7: Analytics & Framework Feedback"""
    dist_plan = state["distribution_plan"]
    brief = state["content_brief"]
    
    user_prompt = f"""DISTRIBUTION: {json.dumps(dist_plan, ensure_ascii=False)[:1500]}...
GOALS: {brief.get('kpis', {})}

Tạo framework. CHỈ trả về JSON."""
    
    response = groq_chat(ANALYST_SYSTEM_PROMPT, user_prompt, temperature=0.5, max_tokens=600)
    
    try:
        analytics = json.loads(response.replace("```json", "").replace("```", "").strip())
    except:
        analytics = {"raw": response}
    
    return {**state, "performance_metrics": analytics.get("kpis", {}),
            "optimization_notes": analytics.get("feedback_for_next_cycle", {}),
            "current_step": "complete"}

print("✅ Cell 9: Analytics Node định nghĩa xong")

✅ Cell 9: Analytics Node định nghĩa xong


In [11]:
# ═══════════════════════════════════════════════════════════════
# CELL 10: ORCHESTRATOR
# ═══════════════════════════════════════════════════════════════
def run_content_factory(client_brief: dict) -> ContentFactoryState:
    """Chạy full pipeline: StoryBrand → PAS → AIDA"""
    
    state = {
        "client_brief": client_brief,
        "content_brief": None, "research_data": None,
        "long_form_draft": None, "social_copies": None,
        "visual_prompts": None, "seo_optimized_content": None,
        "qa_score": None, "qa_feedback": None,
        "approved_content": None, "distribution_plan": None,
        "performance_metrics": None, "optimization_notes": None,
        "current_step": "strategist", "retry_count": 0
    }
    
    print("🚀 CONTENT FACTORY PRO — StoryBrand + PAS + AIDA")
    print("=" * 60)
    
    # Node 1: StoryBrand Strategy
    print("\n📌 [1/7] STORYBRAND STRATEGIST...")
    state = strategist_node(state)
    sb = (state.get('content_brief') or {}).get('storybrand_framework', {})
    print(f"   ✅ Hero: {sb.get('hero', 'N/A')[:50]}...")
    print(f"   ✅ Problem: {sb.get('external_problem', 'N/A')[:50]}...")
    
    # Node 2: Pain Point Research
    print("\n🔍 [2/7] PAIN POINT RESEARCH...")
    state = researcher_node(state)
    research = state.get('research_data') or {}
    print(f"   ✅ {len(research.get('pain_points', []))} pain points")
    print(f"   ✅ {len(research.get('proof_data', {}).get('case_studies', []))} case studies")
    
    # Node 3: PAS + AIDA + StoryBrand Creation
    print("\n✍️ [3/7] PAS + AIDA + STORYBRAND CREATION...")
    state = creation_node(state)
    print(f"   ✅ Draft: {len(state.get('long_form_draft') or '')} chars")
    
    # Node 4: SEO
    print("\n🔍 [4/7] SEO OPTIMIZATION...")
    state = seo_optimizer_node(state)
    seo = state.get('seo_optimized_content') or {}
    print(f"   ✅ Title: {seo.get('title_tag', 'N/A')[:50]}...")
    
    # Node 5: QA — tự động approve, không cần duyệt
    print("\n✅ [5/7] FRAMEWORK COMPLIANCE QA...")
    state = qa_node(state)
    score = state.get('qa_score', 0)
    print(f"   Score: {score}/10 — tự động approve")
    state['approved_content'] = state.get('seo_optimized_content') or {}
    state['current_step'] = 'distribute'
    
    # Node 6: Distribution
    print("\n📤 [6/7] DISTRIBUTION PLANNING...")
    state = distributor_node(state)
    dist = state.get('distribution_plan') or {}
    print(f"   ✅ {len(dist.get('distribution_timeline', []))} posts scheduled")
    
    # Node 7: Analytics
    print("\n📊 [7/7] FRAMEWORK ANALYTICS...")
    state = analytics_node(state)
    opt = state.get('optimization_notes') or {}
    print(f"   ✅ {len(opt.get('storybrand_improvements', []))} StoryBrand insights")
    
    print("\n" + "=" * 60)
    print("✅ WORKFLOW HOÀN TẤT")
    return state

print("✅ Cell 10: Orchestrator định nghĩa xong")

✅ Cell 10: Orchestrator định nghĩa xong


In [12]:
# ═══════════════════════════════════════════════════════════════
# CELL 11: RUN FULL WORKFLOW
# ═══════════════════════════════════════════════════════════════
reset_tracker()                          # reset trước khi chạy
result = run_content_factory(TEST_BRIEF)
print_token_summary()                    # in sau khi xong

🚀 CONTENT FACTORY PRO — StoryBrand + PAS + AIDA

📌 [1/7] STORYBRAND STRATEGIST...
  ✓ +927p +1088c tokens (total 2015)
   ✅ Hero: Founder startup Việt Nam...
   ✅ Problem: Quản lý dự án phức tạp, nhiều công cụ không hiệu q...

🔍 [2/7] PAIN POINT RESEARCH...
  [MOCK] Competitor research: 3 competitors
  [MOCK] RAG retrieved: 4 chunks
  ✓ +527p +661c tokens (total 3203)
   ✅ 8 pain points
   ✅ 1 case studies

✍️ [3/7] PAS + AIDA + STORYBRAND CREATION...
  ✓ +1539p +789c tokens (total 5531)
  ✓ +687p +594c tokens (total 6812)
  ✓ +134p +491c tokens (total 7437)
   ✅ Draft: 2901 chars

🔍 [4/7] SEO OPTIMIZATION...
  ✓ +1000p +600c tokens (total 9037)
   ✅ Title: N/A...

✅ [5/7] FRAMEWORK COMPLIANCE QA...
  ✓ +1151p +247c tokens (total 10435)
   Score: 7.7/10 — tự động approve

📤 [6/7] DISTRIBUTION PLANNING...
  ✓ +174p +372c tokens (total 10981)
   ✅ 6 posts scheduled

📊 [7/7] FRAMEWORK ANALYTICS...
  ✓ +543p +344c tokens (total 11868)
   ✅ 2 StoryBrand insights

✅ WORKFLOW HOÀN TẤT

╔═════

In [13]:
# ═══════════════════════════════════════════════════════════════
# CELL 12: INSPECT OUTPUT DETAIL
# ═══════════════════════════════════════════════════════════════

print("🔍 CHI TIẾT OUTPUT — StoryBrand + PAS + AIDA\n")

# StoryBrand
print("━" * 50)
print("📌 STORYBRAND FRAMEWORK")
print("━" * 50)
sb = result.get("content_brief", {}).get("storybrand_framework", {})
print(f"Hero: {sb.get('hero', 'N/A')}")
print(f"External Problem: {sb.get('external_problem', 'N/A')[:80]}...")
print(f"Internal Problem: {sb.get('internal_problem', 'N/A')[:80]}...")
print(f"Guide: {sb.get('guide', 'N/A')[:80]}...")
print(f"Plan: {sb.get('plan', [])}")
print(f"Direct CTA: {sb.get('direct_cta', 'N/A')}")
print(f"Success Vision: {sb.get('success_vision', 'N/A')[:80]}...")

# PAS
print("\n" + "━" * 50)
print("🔥 PAS FRAMEWORK")
print("━" * 50)
pas = result.get("content_brief", {}).get("pas_framework", {})
print(f"Problem: {pas.get('problem_statement', 'N/A')[:80]}...")
print(f"Agitation: {pas.get('agitation_points', [])[:2]}")

# AIDA in content
print("\n" + "━" * 50)
print("✍️ CONTENT STRUCTURE (AIDA)")
print("━" * 50)
draft = result.get("long_form_draft", "") or ""
print(f"Total length: {len(draft)} chars")
print(f"First 300 chars (Hook):\n{draft[:300]}...")

# QA
print("\n" + "━" * 50)
print("✅ QA FRAMEWORK COMPLIANCE")
print("━" * 50)
print(f"Overall: {result.get('qa_score', 'N/A')}/10")

# Analytics
print("\n" + "━" * 50)
print("📊 FRAMEWORK FEEDBACK")
print("━" * 50)
opt = result.get("optimization_notes", {})
print(f"StoryBrand improvements: {opt.get('storybrand_improvements', [])[:2]}")
print(f"PAS improvements: {opt.get('pas_improvements', [])[:2]}")
print(f"AIDA improvements: {opt.get('aida_improvements', [])[:2]}")

🔍 CHI TIẾT OUTPUT — StoryBrand + PAS + AIDA

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
📌 STORYBRAND FRAMEWORK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Hero: Founder startup Việt Nam
External Problem: Quản lý dự án phức tạp, nhiều công cụ không hiệu quả...
Internal Problem: Lo lắng về việc lãng phí thời gian, không kiểm soát được dự án...
Guide: TechStart Vietnam - Chuyên gia quản lý dự án AI cho startup...
Plan: ['Tối ưu hóa quy trình quản lý dự án', 'Tích hợp AI để tự động hóa công việc', 'Tăng hiệu suất làm việc và giảm thiểu lãng phí thời gian']
Direct CTA: Đăng ký dùng thử ngay hôm nay
Success Vision: Tối ưu hóa thời gian, tăng hiệu suất làm việc và đạt được thành công với startup...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
🔥 PAS FRAMEWORK
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Problem: Quản lý dự án phức tạp và không hiệu quả đang làm lãng phí thời gian và giảm hiệ...
Agitation: ['Lãng phí thời gian vào việc quản lý dự án', 'Không kiểm soá

In [16]:
# ═══════════════════════════════════════════════════════════════
# CELL 13: SAVE RESULT TO FILE
# ═══════════════════════════════════════════════════════════════

output = {
    "timestamp": datetime.now().isoformat(),
    "framework": "StoryBrand + PAS + AIDA Hybrid",
    "client_brief": result.get("client_brief"),
    "storybrand_framework": result.get("content_brief", {}).get("storybrand_framework"),
    "pas_framework": result.get("content_brief", {}).get("pas_framework"),
    "research_data": result.get("research_data"),
    "long_form_draft": result.get("long_form_draft"),
    "social_copies": result.get("social_copies"),
    "visual_prompts": result.get("visual_prompts"),
    "seo_optimized_content": result.get("seo_optimized_content"),
    "qa_score": result.get("qa_score"),
    "qa_feedback": result.get("qa_feedback"),
    "distribution_plan": result.get("distribution_plan"),
    "performance_metrics": result.get("performance_metrics"),
    "optimization_notes": result.get("optimization_notes")
}

import os
output_dir = "./output"
os.makedirs(output_dir, exist_ok=True)
output_path = f"{output_dir}/content_factory_storybrand_pas_aida.json"
with open(output_path, "w", encoding="utf-8") as f:
    json.dump(output, f, ensure_ascii=False, indent=2)